In [ ]:
# the purpose of this notebook is to confirm that the most uncertain formulas are ones outside of the training data 
# formulation space

In [ ]:
# import the data

from pathlib import Path
import pandas as pd

# Load the compiled data
project_root = Path.cwd().resolve().parent if 'notebooks' in str(Path.cwd()) else Path.cwd().resolve()
excel_path = project_root / 'data' / 'raw' / 'Compiled Data.xlsx'

df = pd.read_excel(excel_path)

formula_cols = ["TEM", "Joncryl", "NC513", "PEG-DE", "NGDE"]
measured_target_cols = [
    "Avg_Density_g_cm3",
    "24h_avg_pct absorption",
    "24h_avg_pct_retention",
]
measured_se_cols = [
    "Avg_Density SE",
    "24h_se.1",
    "24h_se",
]

# Keep each target beside its standard error column
measured_cols = [
    col
    for target, se in zip(measured_target_cols, measured_se_cols)
    for col in (target, se)
]
df = df.loc[:, formula_cols + measured_cols]
df = df.rename(columns={
    "24h_se.1": "se_absorption",
    "24h_se": "se_retention",
})

print(df.head())

In [ ]:
# get the 100,000 candidate formulas, they are stored in candidate_set

# Formula constraints:
# - TEM: 3-30%.
# - Each chain extender (Joncryl, NC513, PEG-DE, NGDE): 0.7-9% when included; 0% when omitted.
# - Combined chain extender amount: no more than 20%.
# - Generate 100,000 valid candidate formulas.

import numpy as np
import pandas as pd

candidate_count = 100_000
chain_extender_cols = ["Joncryl", "NC513", "PEG-DE", "NGDE"]
feature_cols = ["TEM", *chain_extender_cols]
rng = np.random.default_rng(42)

candidate_chunks = []
collected = 0
while collected < candidate_count:
    batch_size = max(10_000, 2 * (candidate_count - collected))
    active_counts = rng.integers(1, len(chain_extender_cols) + 1, size=batch_size)
    active_order = rng.random((batch_size, len(chain_extender_cols))).argsort(axis=1)
    active_mask = active_order < active_counts[:, None]
    extender_amounts = rng.uniform(0.7, 9.0, size=(batch_size, len(chain_extender_cols)))
    extender_amounts = np.where(active_mask, extender_amounts, 0.0)

    valid_mask = extender_amounts.sum(axis=1) <= 20.0
    valid_batch = extender_amounts[valid_mask]
    candidate_chunks.append(valid_batch)
    collected += len(valid_batch)

extender_values = np.concatenate(candidate_chunks, axis=0)[:candidate_count]
candidate_set = pd.DataFrame(extender_values, columns=chain_extender_cols)
candidate_set.insert(0, "TEM", rng.uniform(3.0, 30.0, size=candidate_count))
candidate_set = candidate_set[feature_cols]

# Confirm the generated formulas satisfy every requested bound.
assert len(candidate_set) == candidate_count
assert candidate_set["TEM"].between(3.0, 30.0).all()
assert candidate_set[chain_extender_cols].sum(axis=1).le(20.0).all()
for col in chain_extender_cols:
    active = candidate_set[col].ne(0)
    assert candidate_set.loc[active, col].between(0.7, 9.0).all()

print(f"Generated {len(candidate_set):,} valid candidate formulas.")
print(candidate_set.head())
print("Maximum total chain extender:", candidate_set[chain_extender_cols].sum(axis=1).max())

In [ ]:
# get predictions and predictive uncertainty for these formulas, stored in
# candidate_predictions

import joblib
from pathlib import Path

project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
models_dir = project_root / "models"
gpr_models = joblib.load(models_dir / "gpr_models.joblib")

feature_cols = ["TEM", "Joncryl", "NC513", "PEG-DE", "NGDE"]
X_candidates = candidate_set.loc[:, feature_cols]

batch_size = 5_000
prediction_batches = {name: {"mean": [], "std": []} for name in gpr_models}

for name, model in gpr_models.items():
    for start in range(0, len(X_candidates), batch_size):
        batch = X_candidates.iloc[start : start + batch_size]
        mean_batch, std_batch = model.predict(batch, return_std=True)
        prediction_batches[name]["mean"].append(mean_batch)
        prediction_batches[name]["std"].append(std_batch)

prediction_columns = {}
for name, batches in prediction_batches.items():
    prediction_columns[f"pred_{name}"] = np.concatenate(batches["mean"])
    prediction_columns[f"std_{name}"] = np.concatenate(batches["std"])

predictions = pd.DataFrame(prediction_columns, index=candidate_set.index)
candidate_predictions = pd.concat([candidate_set, predictions], axis=1)

required_std_cols = [f"std_{name}" for name in gpr_models]
missing_std_cols = [col for col in required_std_cols if col not in candidate_predictions.columns]
if missing_std_cols:
    raise KeyError(f"Prediction standard-deviation columns were not created: {missing_std_cols}")

display(candidate_predictions.head())


In [ ]:
# find the 10 most uncertain formulas (highest std)